# Differential expression: volcano & MA plots

Simulate two groups of samples across many genes, run a per-gene t-test and visualize the log-fold-change vs significance.

In [ ]:
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
rng = np.random.default_rng(13)
n_genes, n_samples = 4000, 12
base = np.exp(rng.normal(0, 1.2, n_genes))[:, None]
ctrl = rng.gamma(shape=base*5, scale=1/5, size=(n_genes, n_samples//2))
treat = rng.gamma(shape=base*5, scale=1/5, size=(n_genes, n_samples//2))
de = rng.choice(n_genes, 300, replace=False)
treat[de] *= rng.uniform(2, 6, size=(len(de), 1))

In [ ]:
logfc = np.log2((treat.mean(axis=1) + 1e-6) / (ctrl.mean(axis=1) + 1e-6))
pvals = [stats.ttest_ind(ctrl[g], treat[g], equal_var=False).pvalue for g in range(n_genes)]
pvals = np.array(pvals)

fig, axes = plt.subplots(1, 2, figsize=(10, 4.2))
sig = pvals < 0.05/n_genes
axes[0].scatter(logfc, -np.log10(pvals + 1e-12), s=3, alpha=0.5)
axes[0].scatter(logfc[sig], -np.log10(pvals[sig] + 1e-12), s=5, color='#e05b5b')
axes[0].axhline(-np.log10(0.05/n_genes), color='#8b97a5', ls='--')
axes[0].set_xlabel('log2 fold change'); axes[0].set_ylabel('-log10 p')
axes[0].set_title(f'Volcano — {sig.sum()} DE genes (BH)')
meanexpr = np.log2((treat.mean(axis=1) + ctrl.mean(axis=1)) / 2 + 1e-6)
axes[1].scatter(meanexpr, logfc, s=3, alpha=0.5)
axes[1].axhline(0, color='#8b97a5', lw=0.8)
axes[1].set_xlabel('mean expression (log2)'); axes[1].set_ylabel('log2 fold change')
axes[1].set_title('MA plot')